# Generate statistics

## Run this for both HiRES or CHARM experiments

In [1]:
library(tidyverse)
library(ggpubr)
library(yaml)
library(patchwork)

config <- read_yaml(file = "/mnt/ssd/zliu/run_charm/CHARM_preprocess_pipeline/config.yaml")

to_gigabases <- function(raw_bp) {
  raw_bp / 4 * 300 / 1e9
}

extract_sample <- function(path, slice_position, strip_after = NULL) {
  sample <- str_split(path, "/", simplify = TRUE)[, slice_position]
  if (!is.null(strip_after)) {
    sample <- str_split(sample, strip_after, simplify = TRUE)[, 1]
  }
  sample
}

read_read_stat <- function(path, slice_position = 3, strip_after = NULL) {
  read_table2(path, col_names = FALSE) %>%
    arrange(X1) %>%
    mutate(
      sample_id = extract_sample(X1, slice_position, strip_after),
      gigabases = to_gigabases(X2)
    ) %>%
    select(sample_id, gigabases)
}

read_pair_stat <- function(path, slice_position = 3, suffix_to_remove = NULL) {
  read_table2(path, col_names = FALSE) %>%
    arrange(X1) %>%
    mutate(
      sample_id = extract_sample(X1, slice_position),
      sample_id = if (!is.null(suffix_to_remove)) str_remove(sample_id, fixed(suffix_to_remove)) else sample_id
    ) %>%
    select(sample_id, pairs = X2)
}

raw_reads <- read_read_stat("../stat/raw.fq.stat", strip_after = "_") %>%
  rename(Rawreads = gigabases)
dna_reads <- read_read_stat("../stat/dna.fq.stat") %>%
  rename(DNAreads = gigabases)
rna_reads <- read_read_stat("../stat/rna.fq.stat") %>%
  rename(RNAreads = gigabases)

raw_pairs <- read_pair_stat("../stat/raw.pairs.stat") %>%
  rename(raw_pairs = pairs)
pairs_dedup <- read_pair_stat("../stat/pairs.dedup.stat") %>%
  rename(pairs_dedup = pairs)

pairs_clean1 <- read_pair_stat("../stat/pairs.c1.stat", slice_position = 5, suffix_to_remove = ".pairs.gz") %>%
  rename(pairs_clean1 = pairs)
pairs_clean2 <- read_pair_stat("../stat/pairs.c12.stat", slice_position = 5, suffix_to_remove = ".pairs.gz") %>%
  rename(pairs_clean2 = pairs)
pairs_clean3 <- read_pair_stat("../stat/pairs.c123.stat", slice_position = 5, suffix_to_remove = ".pairs.gz") %>%
  rename(pairs_clean3 = pairs)
inter_pairs_clean3 <- read_pair_stat("../stat/inter.pairs.c123.stat", slice_position = 5, suffix_to_remove = ".pairs.gz") %>%
  rename(inter_pairs_clean3 = pairs)

yperx <- read_table2("../stat/yperx.stat", col_names = FALSE) %>%
  arrange(X1) %>%
  mutate(sample_id = extract_sample(X1, 2)) %>%
  select(sample_id, yperx = X2)

stat <- raw_reads %>%
  left_join(dna_reads, by = "sample_id") %>%
  left_join(rna_reads, by = "sample_id") %>%
  left_join(yperx, by = "sample_id") %>%
  left_join(raw_pairs, by = "sample_id") %>%
  left_join(pairs_dedup, by = "sample_id") %>%
  left_join(pairs_clean1, by = "sample_id") %>%
  left_join(pairs_clean2, by = "sample_id") %>%
  left_join(pairs_clean3, by = "sample_id") %>%
  left_join(inter_pairs_clean3, by = "sample_id")

rna_gene_counts <- read_table2("../result/RNA_Res/counts.gene.total.format.tsv")
rna_gene_matrix <- as.data.frame(rna_gene_counts %>% select(-gene))
feature_stat_gene <- tibble(
  sample_id = names(rna_gene_matrix),
  UMIs_gene = colSums(rna_gene_matrix),
  genes_gene = colSums(rna_gene_matrix != 0)
)

# RNA per-cell stats: annotation rate and dedup rate
rna_reads_path <- "../stat/rna.reads_per_cell.stat"
if (file.exists(rna_reads_path)) {
  rna_per_cell <- read_tsv(rna_reads_path, col_names = c("sample_id", "rna_total_mapped_reads", "rna_assigned_reads"),
                           col_types = "cdd")
  feature_stat_gene <- feature_stat_gene %>%
    left_join(rna_per_cell, by = "sample_id") %>%
    mutate(
      rna_annotation_rate = ifelse(rna_total_mapped_reads > 0,
                                   rna_assigned_reads / rna_total_mapped_reads * 100,
                                   NA_real_),
      rna_dedup_rate = ifelse(rna_assigned_reads > 0,
                              (1 - UMIs_gene / rna_assigned_reads) * 100,
                              NA_real_)
    )
}

# RNA DNA contamination rate (GATC pattern in clean R2 reads)
rna_contam_path <- "../stat/rna.dna_contam.stat"
if (file.exists(rna_contam_path)) {
  rna_contam <- read_tsv(rna_contam_path, col_names = c("sample_id", "rna_clean_reads", "rna_gatc_reads"),
                         col_types = "cdd")
  feature_stat_gene <- feature_stat_gene %>%
    left_join(rna_contam, by = "sample_id") %>%
    mutate(
      rna_dna_contam_rate = ifelse(rna_clean_reads > 0,
                                   rna_gatc_reads / rna_clean_reads * 100,
                                   NA_real_)
    )
}

rna_exon_counts <- read_table2("../result/RNA_Res/counts.exon.total.format.tsv")
rna_exon_matrix <- as.data.frame(rna_exon_counts %>% select(-gene))
feature_stat_exon <- tibble(
  sample_id = names(rna_exon_matrix),
  UMIs_exon = colSums(rna_exon_matrix),
  genes_exon = colSums(rna_exon_matrix != 0)
)

if (config$if_RNA_snp_split) {
  genome1_counts <- read_table2("../result/RNA_Res/counts.gene.genome1.tsv")
  genome1_matrix <- as.data.frame(genome1_counts %>% select(-gene))
  genome2_counts <- read_table2("../result/RNA_Res/counts.gene.genome2.tsv")
  genome2_matrix <- as.data.frame(genome2_counts %>% select(-gene))

  feature_stat_genome1 <- tibble(
    sample_id = names(genome1_matrix),
    UMIs_gene_genome1 = colSums(genome1_matrix),
    genes_gene_genome1 = colSums(genome1_matrix != 0)
  )

  feature_stat_genome2 <- tibble(
    sample_id = names(genome2_matrix),
    UMIs_gene_genome2 = colSums(genome2_matrix),
    genes_gene_genome2 = colSums(genome2_matrix != 0)
  )

  stat <- stat %>%
    left_join(feature_stat_gene, by = "sample_id") %>%
    left_join(feature_stat_exon, by = "sample_id") %>%
    left_join(feature_stat_genome1, by = "sample_id") %>%
    left_join(feature_stat_genome2, by = "sample_id")
} else {
  stat <- stat %>%
    left_join(feature_stat_gene, by = "sample_id") %>%
    left_join(feature_stat_exon, by = "sample_id")
}

stat <- stat %>%
  rename(cellname = sample_id)

if (config$if_structure) {
  rmsd <- read_lines("../stat/rmsd.info") %>%
    tibble(line = .) %>%
    mutate(
      value = as.numeric(str_extract(line, "[-+]?[0-9]*\\.?[0-9]+(?:[eE][-+]?[0-9]+)?$")),
      path = str_extract(line, ".*(?=:\\s*\\[M::__main__\\])"),
      path = if_else(is.na(path), str_extract(line, "^[^\\s]+"), path)
    ) %>%
    filter(!is.na(path), !is.na(value)) %>%
    mutate(
      m = str_match(path, ".*/3d_info/([^/]+)/[^/]*\\.([^.]+)\\.align\\.rms\\.info$"),
      cellname = m[, 2],
      resolution = m[, 3]
    ) %>%
    filter(!is.na(cellname), !is.na(resolution)) %>%
    select(cellname, resolution, rmsd = value) %>%
    distinct() %>%
    pivot_wider(names_from = resolution, values_from = rmsd, names_prefix = "rmsd_") %>%
    arrange(cellname)

  stat <- stat %>% left_join(rmsd, by = "cellname")
}

if (config$if_charm) {
  charm_reads <- map(
    names(config$split),
    function(split_name) {
      read_csv(paste0("../stat/", split_name, ".read.stat"), col_names = FALSE) %>%
        transmute(
          cellname = X1,
          !!paste0(split_name, "_reads") := X2 / 2 * 300 / 1e9
        )
    }
  ) %>%
    reduce(full_join, by = "cellname")

  stat <- stat %>% full_join(charm_reads, by = "cellname")

  # ATAC/CUT&Tag per-cell dedup rates
  charm_dedup <- map(
    names(config$split),
    function(split_name) {
      path <- paste0("../stat/", split_name, ".dedup_rate.stat")
      if (file.exists(path)) {
        read_tsv(path, col_names = c("cellname", paste0(split_name, "_dedup_rate")),
                 col_types = "cd")
      } else {
        tibble(cellname = character())
      }
    }
  ) %>% reduce(full_join, by = "cellname")

  stat <- stat %>% full_join(charm_dedup, by = "cellname")
}

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
“`read_table2()` was deprecated in readr 2.0.0.
ℹ Please use `read_table()` instead.”

── Column specification ────────────────────────────────────────────────────────
cols(
  X1 = col_character(),
  X2 = col_double()
)


── Column specification ────────────────────────────────────────────────────────
cols(
  X1 = col_character(),
  X2 = col_double()
)


── Column specification ────────────────────────────────────────────────────────
cols(
  X1 = col_char

In [2]:
fill_numeric <- function(x) {
  x[is.na(x) | is.nan(x)] <- 0
  x
}

qc_metrics <- stat %>%
  mutate(
    RNAreadsRatio = RNAreads / (RNAreads + DNAreads),
    pairsPerRead = raw_pairs / DNAreads / 1e9 * 300,
    pairsValidRatio = pairs_clean3 / raw_pairs,
    interPairsRatio = inter_pairs_clean3 / pairs_clean3
  ) %>%
  mutate(across(where(is.numeric), fill_numeric))


In [3]:
qc_metrics


cellname,Rawreads,DNAreads,RNAreads,yperx,raw_pairs,pairs_dedup,pairs_clean1,pairs_clean2,pairs_clean3,⋯,rmsd_20k,rmsd_50k,ct_reads,atac_reads,ct_dedup_rate,atac_dedup_rate,RNAreadsRatio,pairsPerRead,pairsValidRatio,interPairsRatio
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
R1P1008,5.527111,5.526642,0.0003981,0.002528,5624735,983003,967750,881681,881338,⋯,0.3104733,0.16539232,0.00222210,0.00563085,43.44,35.83,7.202771e-05,0.3053248,0.15668969,0.2645659
R1P1028,5.206169,5.204944,0.0010806,0.003271,5300813,851069,836428,744941,744687,⋯,0.2669561,0.12000285,0.00201675,0.00613035,46.51,39.23,2.075672e-04,0.3055256,0.14048543,0.2319444
R1P1038,5.654884,5.654345,0.0004536,0.004422,5135487,726581,713587,629732,629536,⋯,0.1771268,0.06775184,0.00105360,0.00162120,51.01,43.81,8.021505e-05,0.2724712,0.12258545,0.2061328
R2P1008,7.808585,7.805679,0.0026370,0.004446,8664824,1157430,1138066,1068465,1067965,⋯,0.4227675,0.18476054,0.00365730,0.01063080,52.59,42.72,3.377169e-04,0.3330200,0.12325294,0.2954732
R2P1028,3.779242,3.778127,0.0009963,0.003704,3717999,660534,651757,564067,563845,⋯,0.3695226,0.16551952,0.00113250,0.00260865,45.30,37.27,2.636326e-04,0.2952256,0.15165281,0.1998031
R2P1038,3.305252,3.304431,0.0007410,0.004111,3533154,850500,841381,753432,753006,⋯,1.4179784,0.96260868,0.00138075,0.00213150,36.64,32.08,2.241941e-04,0.3207651,0.21312572,0.1740995
R3P1008,10.568173,10.567372,0.0006693,0.005260,11647297,1184323,1156957,1085535,1085166,⋯,0.2531795,0.12495770,0.00485685,0.00938100,60.65,51.05,6.333246e-05,0.3306583,0.09316891,0.3308885
R3P1028,6.575642,6.573643,0.0017670,0.003248,7146757,839393,824648,729884,729509,⋯,0.5460890,0.26930379,0.00387675,0.01396830,55.35,44.51,2.687285e-04,0.3261551,0.10207553,0.2301822
R3P1038,4.199046,4.196801,0.0020199,0.005564,4525282,679520,671471,581500,581160,⋯,0.6362367,0.32531559,0.00689580,0.00929940,47.04,37.72,4.810636e-04,0.3234808,0.12842515,0.1822923


## Run below if this is a CHARM experiment

In [4]:
suppressPackageStartupMessages({
  library(Signac)
  library(Seurat)
  library(EnsDb.Mmusculus.v79)
  library(BSgenome.Mmusculus.UCSC.mm10)
  library(future)
})

plan("multicore", workers = 10)


In [5]:
charm <- rna_gene_counts %>%
  column_to_rownames("gene") %>%
  as.matrix() %>%
  CreateSeuratObject(assay = "rna", min.cells = 0, min.features = 0)


Warning message:
"Data is of class matrix. Coercing to dgCMatrix."


In [6]:
cell_names <- intersect(
  rownames(charm@meta.data),
  colnames(rna_gene_counts %>% dplyr::select(-gene))
)


In [7]:
atac_fragments <- CreateFragmentObject("../result/fragments/atac.fragments.bgz", cells = cell_names)
ct_fragments <- CreateFragmentObject("../result/fragments/ct.fragments.bgz", cells = cell_names)
mm10_genome <- seqlengths(BSgenome.Mmusculus.UCSC.mm10)
atac_count_matrix <- GenomeBinMatrix(atac_fragments, binsize = 5000, genome = mm10_genome)
ct_count_matrix <- GenomeBinMatrix(ct_fragments, binsize = 5000, genome = mm10_genome)

atac_assay <- CreateChromatinAssay(counts = atac_count_matrix, fragments = atac_fragments, genome = "mm10")
ct_assay <- CreateChromatinAssay(counts = ct_count_matrix, fragments = ct_fragments, genome = "mm10")

charm[["atac"]] <- atac_assay
charm[["ct"]] <- ct_assay


Computing hash



Computing hash

Warning message in SingleFeatureMatrix(fragment = fragments[[x]], features = features, :
"18798 features are on seqnames not present in the fragment file. These will be removed."
Extracting reads overlapping genomic regions

Warning message in SingleFeatureMatrix(fragment = fragments[[x]], features = features, :
"18798 features are on seqnames not present in the fragment file. These will be removed."
Extracting reads overlapping genomic regions



In [8]:
charm@meta.data <- charm@meta.data %>%
  rownames_to_column("cellname")
rownames(charm@meta.data) <- charm@meta.data$cellname


In [9]:
options(future.globals.maxSize = 10 * 1024^3)
# calc tss enrichment
annotations <- GetGRangesFromEnsDb(ensdb = EnsDb.Mmusculus.v79)
seqlevelsStyle(annotations) <- "UCSC"
genome(annotations) <- "mm10"

Annotation(charm[["atac"]]) <- annotations
Annotation(charm[["ct"]]) <- annotations

charm <- TSSEnrichment(charm, assay = "atac", fast = FALSE)
charm@meta.data$TSS.enrichment.atac <- charm@meta.data$TSS.enrichment

charm <- TSSEnrichment(charm, assay = "ct", fast = FALSE)
charm@meta.data$TSS.enrichment.ct <- charm@meta.data$TSS.enrichment
charm@meta.data$TSS.enrichment <- NULL

qc_metrics <- qc_metrics %>% full_join(
  charm@meta.data %>% dplyr::select(cellname, nCount_atac, nCount_ct, TSS.enrichment.atac, TSS.enrichment.ct),
  by = "cellname"
)


Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warn

In [10]:
options(repr.matrix.max.cols = 100)
qc_metrics

cellname,Rawreads,DNAreads,RNAreads,yperx,raw_pairs,pairs_dedup,pairs_clean1,pairs_clean2,pairs_clean3,inter_pairs_clean3,UMIs_gene,genes_gene,rna_total_mapped_reads,rna_assigned_reads,rna_annotation_rate,rna_dedup_rate,rna_clean_reads,rna_gatc_reads,rna_dna_contam_rate,UMIs_exon,genes_exon,rmsd_1m,rmsd_200k,rmsd_20k,rmsd_50k,ct_reads,atac_reads,ct_dedup_rate,atac_dedup_rate,RNAreadsRatio,pairsPerRead,pairsValidRatio,interPairsRatio,nCount_atac,nCount_ct,TSS.enrichment.atac,TSS.enrichment.ct
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
R1P1008,5.527111,5.526642,0.0003981,0.002528,5624735,983003,967750,881681,881338,233172,720,488,1298,931,71.72573,22.66380,1327,172,12.961567,133,111,1.613187e-02,0.04074786,0.3104733,0.16539232,0.00222210,0.00563085,43.44,35.83,7.202771e-05,0.3053248,0.15668969,0.2645659,9910,3369,1.950431,0.9842010
R1P1028,5.206169,5.204944,0.0010806,0.003271,5300813,851069,836428,744941,744687,172726,1960,1133,3599,2721,75.60433,27.96766,3602,259,7.190450,298,234,6.413977e-05,0.01826106,0.2669561,0.12000285,0.00201675,0.00613035,46.51,39.23,2.075672e-04,0.3055256,0.14048543,0.2319444,9969,2965,2.297702,1.2387612
R1P1038,5.654884,5.654345,0.0004536,0.004422,5135487,726581,713587,629732,629536,129768,766,559,1499,1132,75.51701,32.33216,1512,107,7.076720,213,146,3.477263e-04,0.01140300,0.1771268,0.06775184,0.00105360,0.00162120,51.01,43.81,8.021505e-05,0.2724712,0.12258545,0.2061328,2421,1365,3.492803,0.6455083
R2P1008,7.808585,7.805679,0.0026370,0.004446,8664824,1157430,1138066,1068465,1067965,315555,4836,2081,8740,6987,79.94279,30.78574,8790,527,5.995449,781,530,3.941899e-03,0.02930955,0.4227675,0.18476054,0.00365730,0.01063080,52.59,42.72,3.377169e-04,0.3330200,0.12325294,0.2954732,16668,4825,2.211947,1.0753952
R2P1028,3.779242,3.778127,0.0009963,0.003704,3717999,660534,651757,564067,563845,112658,1937,1085,3270,2644,80.85627,26.73979,3321,197,5.931948,311,239,6.465539e-03,0.02992163,0.3695226,0.16551952,0.00113250,0.00260865,45.30,37.27,2.636326e-04,0.2952256,0.15165281,0.1998031,4515,1696,3.056943,1.1373242
R2P1038,3.305252,3.304431,0.0007410,0.004111,3533154,850500,841381,753432,753006,131098,1520,935,2521,1958,77.66759,22.36977,2470,144,5.829960,223,171,1.575000e-01,0.43548614,1.4179784,0.96260868,0.00138075,0.00213150,36.64,32.08,2.241941e-04,0.3207651,0.21312572,0.1740995,4038,2402,2.893403,1.4860140
R3P1008,10.568173,10.567372,0.0006693,0.005260,11647297,1184323,1156957,1085535,1085166,359069,1062,709,2266,1641,72.41836,35.28336,2231,166,7.440610,251,169,8.588137e-05,0.02519249,0.2531795,0.12495770,0.00485685,0.00938100,60.65,51.05,6.333246e-05,0.3306583,0.09316891,0.3308885,12330,5187,2.809055,1.0570075
R3P1028,6.575642,6.573643,0.0017670,0.003248,7146757,839393,824648,729884,729509,167920,3074,1640,5858,4418,75.41823,30.42100,5890,497,8.438031,454,357,1.712220e-04,0.05496590,0.5460890,0.26930379,0.00387675,0.01396830,55.35,44.51,2.687285e-04,0.3261551,0.10207553,0.2301822,21129,4705,2.036262,1.5142752
R3P1038,4.199046,4.196801,0.0020199,0.005564,4525282,679520,671471,581500,581160,105941,3913,1843,6729,5299,78.74870,26.15588,6733,385,5.718105,534,442,7.333494e-03,0.07130193,0.6362367,0.32531559,0.00689580,0.00929940,47.04,37.72,4.810636e-04,0.3234808,0.12842515,0.1822923,16047,10216,2.640658,1.4203978


In [11]:
qc_metrics %>% write_tsv("metadata_raw.tsv")